# Self-Correct Agent Demo

This notebook mirrors `examples/demo.py` and runs without API keys by using a mocked OpenAI-compatible client.

In [ ]:
from unittest.mock import MagicMock

from self_correct import AntiHallucinator

def response(content, prompt_tokens=0, completion_tokens=0):
    mock = MagicMock()
    mock.choices[0].message.content = content
    mock.usage.prompt_tokens = prompt_tokens
    mock.usage.completion_tokens = completion_tokens
    return mock

client = MagicMock()
client.chat.completions.create.side_effect = [
    response('Moon is cheese. Water boils at 100C.', 40, 20),
    response('1. Moon is cheese.\n2. Water boils at 100C.', 30, 10),
    response('VERIFIED: False. The moon is not cheese.', 25, 10),
    response('VERIFIED: True.', 20, 5),
    response('Moon is rock. Water boils at 100C.', 45, 20),
]

agent = AntiHallucinator(client=client, strictness=1.0)
result = agent.generate(model='gpt-4o-mini', prompt='Write two short facts about the moon and water.')

print(result.content)
print('claims flagged:', len(result.hallucinations_caught))
print('total tokens:', result.token_usage.total_tokens)